In [1]:
import mne

In [2]:
file_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_eeg-epo.fif"
epochs = mne.read_epochs(file_path, preload=True)

Reading D:\Users\windows\Desktop\Inner Speech Decoding\Dataset\sub-01_ses-01_eeg-epo.fif ...
    Found the data of interest:
        t =    -500.00 ...    4000.00 ms
        0 CTF compensation matrices available
Not setting metadata
54 matching events found
No baseline correction applied
0 projection items activated


In [3]:
# # 이벤트 ID(클래스) 확인
print(epochs.events[:, -1])

[34 31 34 34 33 32 34 32 34 32 31 31 34 33 32 31 31 34 32 34 32 33 32 31
 33 33 32 31 33 33 34 31 32 33 33 33 34 31 31 32 32 32 34 34 34 31 33 34
 34 32 31 32 34 34]


In [5]:
# 클래스 종류 확인
print(set(epochs.events[:, -1]))

{32, 33, 34, 31}


In [7]:
print(epochs.event_id)

{'arriba': 31, 'abajo': 32, 'derecha': 33, 'izquierda': 34}


In [9]:
# 각 클래스의 epoch 개수 출력
import numpy as np

labels = epochs.events[:, -1]
unique_labels = np.unique(labels)

for label in unique_labels:
    count = np.sum(labels == label)
    print(f"Class {label}: {count} epochs")

Class 31: 12 epochs
Class 32: 14 epochs
Class 33: 11 epochs
Class 34: 17 epochs


In [10]:
from mne.decoding import CSP
import numpy as np

labels = epochs.events[:, -1]
unique_labels = np.unique(labels)

csp_dict = {}

for target_label in unique_labels:
    y_binary = (labels == target_label).astype(int)
    print(f"Class {target_label}: positive samples = {np.sum(y_binary == 1)}, negative samples = {np.sum(y_binary == 0)}")
    
    X = epochs.get_data()
    
    # reg=0.1 또는 'ledoit_wolf' 꼭 넣어보세요!
    csp = CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)
    
    try:
        csp.fit(X, y_binary)
        csp_dict[target_label] = csp
        print(f"Trained CSP for class {target_label} (One-vs-Rest)")
    except Exception as e:
        print(f"Failed to train CSP for class {target_label}: {e}")

Class 31: positive samples = 12, negative samples = 42
Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Trained CSP for class 31 (One-vs-Rest)
Class 32: positive samples = 14, negative samples = 40
Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Trained CSP for class 32 (One-vs-Rest)
Class 33: positive samples = 11, negative samples = 43
Computing rank from data with rank=No

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score
import numpy as np

X = epochs.get_data()  # (n_epochs, n_channels, n_times)
y = labels

unique_labels = np.unique(y)
all_accuracies = []

for target_label in unique_labels:
    print(f'\nClass {target_label} vs Rest')

    # One-vs-Rest 라벨 생성
    y_binary = (y == target_label).astype(int)

    # CSP 객체 불러오기
    csp = csp_dict[target_label]
    X_csp = csp.transform(X)

    # 5-Fold 교차 검증
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accuracies = []

    for train_idx, test_idx in skf.split(X_csp, y_binary):
        X_train, X_test = X_csp[train_idx], X_csp[test_idx]
        y_train, y_test = y_binary[train_idx], y_binary[test_idx]

        clf = LDA()
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        fold_accuracies.append(acc)

    mean_acc = np.mean(fold_accuracies)
    all_accuracies.append(mean_acc)
    print(f'Accuracy for class {target_label} vs Rest: {mean_acc:.2f}')

print(f'\nOverall Average Accuracy: {np.mean(all_accuracies):.2f}')



Class 31 vs Rest
Accuracy for class 31 vs Rest: 0.76

Class 32 vs Rest
Accuracy for class 32 vs Rest: 0.93

Class 33 vs Rest
Accuracy for class 33 vs Rest: 0.78

Class 34 vs Rest
Accuracy for class 34 vs Rest: 0.70

Overall Average Accuracy: 0.79


In [12]:
from sklearn.model_selection import LeaveOneOut
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score
import numpy as np

X = epochs.get_data()  # (n_epochs, n_channels, n_times)
y = labels

unique_labels = np.unique(y)
all_accuracies = []

for target_label in unique_labels:
    print(f'\nClass {target_label} vs Rest')

    # One-vs-Rest 라벨 생성
    y_binary = (y == target_label).astype(int)

    # CSP 객체 불러오기
    csp = csp_dict[target_label]
    X_csp = csp.transform(X)

    # Leave-One-Out CV 준비
    loo = LeaveOneOut()
    fold_accuracies = []

    for train_idx, test_idx in loo.split(X_csp, y_binary):
        X_train, X_test = X_csp[train_idx], X_csp[test_idx]
        y_train, y_test = y_binary[train_idx], y_binary[test_idx]

        clf = LDA()
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        fold_accuracies.append(acc)

    mean_acc = np.mean(fold_accuracies)
    all_accuracies.append(mean_acc)
    print(f'Accuracy for class {target_label} vs Rest: {mean_acc:.2f}')

print(f'\nOverall Average Accuracy: {np.mean(all_accuracies):.2f}')



Class 31 vs Rest
Accuracy for class 31 vs Rest: 0.76

Class 32 vs Rest
Accuracy for class 32 vs Rest: 0.93

Class 33 vs Rest
Accuracy for class 33 vs Rest: 0.76

Class 34 vs Rest
Accuracy for class 34 vs Rest: 0.67

Overall Average Accuracy: 0.78


In [13]:
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score
import numpy as np

# 데이터 준비
X = epochs.get_data()
y = labels

# CSP 객체 (다중 클래스용) 생성
from mne.decoding import CSP

csp = CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)

# CSP 학습 (다중 클래스 라벨 그대로 사용)
csp.fit(X, y)
X_csp = csp.transform(X)

# 5-Fold 교차 검증
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

for train_idx, test_idx in skf.split(X_csp, y):
    X_train, X_test = X_csp[train_idx], X_csp[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf = LDA()
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

print(f'\nMulti-Class Classification Average Accuracy: {np.mean(accuracies):.2f}')


Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=31 covariance using LEDOIT_WOLF
Done.
Estimating class=32 covariance using LEDOIT_WOLF
Done.
Estimating class=33 covariance using LEDOIT_WOLF
Done.
Estimating class=34 covariance using LEDOIT_WOLF
Done.

Multi-Class Classification Average Accuracy: 0.43


In [14]:
######################## CSP + LDA 코드
from sklearn.model_selection import LeaveOneOut
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import accuracy_score

X = epochs.get_data()
y = labels

# CSP 객체 생성
from mne.decoding import CSP
csp = CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)

# CSP 학습
csp.fit(X, y)
X_csp = csp.transform(X)

# Leave-One-Out 교차 검증
loo = LeaveOneOut()
accuracies = []

for train_idx, test_idx in loo.split(X_csp, y):
    X_train, X_test = X_csp[train_idx], X_csp[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    clf = LDA()
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)

print(f'\nMulti-Class CSP + LDA (Leave-One-Out) Accuracy: {np.mean(accuracies):.2f}')


Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=31 covariance using LEDOIT_WOLF
Done.
Estimating class=32 covariance using LEDOIT_WOLF
Done.
Estimating class=33 covariance using LEDOIT_WOLF
Done.
Estimating class=34 covariance using LEDOIT_WOLF
Done.

Multi-Class CSP + LDA (Leave-One-Out) Accuracy: 0.31


In [16]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from mne.decoding import CSP

# 1. 원본 labels와 unique_labels
labels = epochs.events[:, -1]
unique_labels = np.unique(labels)

# 2. 0부터 시작하는 인덱스로 변환하는 매핑 생성
label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
labels_idx = np.array([label_to_idx[label] for label in labels])

n_classes = len(unique_labels)
n_epochs = epochs.get_data().shape[0]

# 3. One-vs-Rest CSP 학습
X = epochs.get_data()  # (n_epochs, n_channels, n_times)
csp_dict = {}
n_components_per_csp = 4

for target_label in unique_labels:
    y_binary = (labels == target_label).astype(int)
    print(f"Class {target_label}: positive samples = {np.sum(y_binary==1)}, negative samples = {np.sum(y_binary==0)}")
    csp = CSP(n_components=n_components_per_csp, reg='ledoit_wolf', log=True, norm_trace=False)
    try:
        csp.fit(X, y_binary)
        csp_dict[target_label] = csp
        print(f"Trained CSP for class {target_label} (One-vs-Rest)")
    except Exception as e:
        print(f"Failed to train CSP for class {target_label}: {e}")

# 4. CSP 변환 후 특징 합치기
csp_features = []
for target_label in unique_labels:
    csp = csp_dict[target_label]
    X_csp = csp.transform(X)  # (n_epochs, n_components_per_csp)
    csp_features.append(X_csp)

X_csp_all = np.concatenate(csp_features, axis=1)  # (n_epochs, n_components_per_csp * n_classes)
X_csp_all = np.expand_dims(X_csp_all, axis=1)     # (n_epochs, 1, features)

# 5. Tensor 변환 (레이블은 0부터 시작하는 labels_idx 사용)
X_tensor = torch.tensor(X_csp_all, dtype=torch.float32)
y_tensor = torch.tensor(labels_idx, dtype=torch.long)

# 6. 1D CNN 모델 정의
class CSPNet1D(nn.Module):
    def __init__(self, input_length, n_classes):
        super(CSPNet1D, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(16)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(32 * input_length, 64)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, n_classes)
        
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# 7. train/test split
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, stratify=labels_idx, random_state=42)
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# 8. 모델 생성 및 학습 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CSPNet1D(input_length=X_csp_all.shape[2], n_classes=n_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# 9. 학습 루프
n_epochs = 30
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{n_epochs} Loss: {total_loss/len(train_loader):.4f}")

# 10. 평가
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(targets.cpu())

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
acc = accuracy_score(all_targets, all_preds)
print(f"Test Accuracy: {acc:.4f}")

Class 31: positive samples = 12, negative samples = 42
Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Trained CSP for class 31 (One-vs-Rest)
Class 32: positive samples = 14, negative samples = 40
Computing rank from data with rank=None
    Using tolerance 31 (2.2e-16 eps * 129 dim * 1.1e+15  max singular value)
    Estimated rank (data): 129
    data: rank 129 computed from 129 data channels with 0 projectors
Reducing data rank from 129 -> 129
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.
Trained CSP for class 32 (One-vs-Rest)
Class 33: positive samples = 11, negative samples = 43
Computing rank from data with rank=No